# Event-Episode v2 — Colab Runner

End-to-end pipeline for the v2 event-episode experiment.  Runs:

1. **Tabular HPO** (Phase 4) — LR/RF/XGB sweep over (T, L, W).
2. **DL HPO + multi-seed** (Phase 5) — GRU/LSTM/RNN/CNN.
3. **Sensor ablation** (Phase 6).
4. **Leakage probe** (Phase 7) — gate before final test.
5. **Final test evaluation** (Phase 8).

All outputs are written under `outputs/v2/tables/`.  This notebook does **not** modify any v1 file.

## 1. Clone / pull repo, install deps, mount Drive

In [1]:
from pathlib import Path

REPO_URL = "https://github.com/bahaa1515/EECE-693-project.git"
REPO_DIR = Path("/content/EECE-693-project")
BRANCH = "codex/tarek-event-episode-tuning"

if REPO_DIR.exists():
    %cd /content/EECE-693-project
    !git fetch --all --quiet
    !git checkout {BRANCH}
    !git pull --ff-only
else:
    %cd /content
    !git clone {REPO_URL} EECE-693-project
    %cd /content/EECE-693-project
    !git checkout {BRANCH}

/content
Cloning into 'EECE-693-project'...
remote: Enumerating objects: 342, done.
remote: Counting objects: 100% (117/117), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 342 (delta 26), reused 106 (delta 23), pack-reused 225 (from 1)
Receiving objects: 100% (342/342), 46.73 MiB | 11.37 MiB/s, done.
Resolving deltas: 100% (99/99), done.
Updating files: 100% (76/76), done.
/content/EECE-693-project
Branch 'codex/tarek-event-episode-tuning' set up to track remote branch 'codex/tarek-event-episode-tuning' from 'origin'.
Switched to a new branch 'codex/tarek-event-episode-tuning'


In [2]:
!pip install -q -r requirements-colab.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 861.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.9/153.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121

In [3]:
import sys, torch
print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

Python: 3.12.13
Torch: 2.4.0+cu121
CUDA: False 


In [12]:
# Mount Drive and point at the AAMOS dataset / output folder.
import os
from pathlib import Path
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("Drive mount skipped:", exc)

data_dir = Path(os.environ.get("AAMOS_DATA_DIR", "/content/drive/MyDrive/AAMOS/dataset"))
out_dir  = Path(os.environ.get("AAMOS_OUTPUT_DIR", "/content/drive/MyDrive/AAMOS/outputs"))
if data_dir.exists():
    os.environ["AAMOS_DATA_DIR"] = str(data_dir)
    os.environ["AAMOS_OUTPUT_DIR"] = str(out_dir)
    print("AAMOS_DATA_DIR =", os.environ["AAMOS_DATA_DIR"])
    print("AAMOS_OUTPUT_DIR =", os.environ["AAMOS_OUTPUT_DIR"])
else:
    print("Set AAMOS_DATA_DIR to the Drive folder containing the AAMOS CSVs.")

Mounted at /content/drive
Set AAMOS_DATA_DIR to the Drive folder containing the AAMOS CSVs.


## 2. Configure the experiment

Set the (T, L, W) grid here.  Smaller grids run faster on Colab CPUs;
the defaults match the plan.

In [5]:
THRESHOLDS = "2,3,4"
LENGTHS    = "3,7,14"      # L=28 omitted for DL (plan); tabular tolerates it
WASHOUTS   = "0,7,14"
SEED       = 42
DL_EPOCHS  = 30
DL_BATCH   = 32
N_SHUFFLES = 5
MULTI_SEEDS = "42,43,44,45,46"

## 3. Phase 4 — Tabular HPO

In [6]:
!python -m scripts.v2.tune_tabular_v2 \
    --thresholds {THRESHOLDS} \
    --lengths {LENGTHS} \
    --washouts {WASHOUTS} \
    --seed {SEED}

[v2-tune-tabular] outputs -> /content/EECE-693-project/outputs/v2/tables
  thresholds=[2, 3, 4]  lengths=[3, 7, 14]  washouts=[0, 7, 14]
  algos=['lr', 'rf', 'xgb']  seed=42  sensor_tag=all

[1/4] Building event labels...

[2/4] Building v2 sample indexes...

[3/4] Building v2 feature tables...

[4/4] Running HPO loop...
  split (seed=42): train_users=(939, 294, 113, 808, 343, 562, 917, 278, 328, 867, 190, 701, 454) | val_users=(625, 514, 398) | test_users=(764, 473, 702, 447)
  T=2 L=3 W=0 algo=lr: 8 trials  (train=963 val=217)
  T=2 L=3 W=0 algo=rf: 24 trials  (train=963 val=217)
  T=2 L=3 W=0 algo=xgb: 32 trials  (train=963 val=217)
  T=2 L=3 W=7 algo=lr: 8 trials  (train=867 val=217)
  T=2 L=3 W=7 algo=rf: 24 trials  (train=867 val=217)
  T=2 L=3 W=7 algo=xgb: 32 trials  (train=867 val=217)
  T=2 L=3 W=14 algo=lr: 8 trials  (train=783 val=217)
  T=2 L=3 W=14 algo=rf: 24 trials  (train=783 val=217)
  T=2 L=3 W=14 algo=xgb: 32 trials  (train=783 val=217)
  T=2 L=7 W=0 algo=lr: 8 tria

In [7]:
import pandas as pd
best = pd.read_csv("outputs/v2/tables/tune_tabular_v2_best.csv")
best.sort_values("val_pr_auc", ascending=False).head(20)

,sensor_tag,threshold,input_length_days,washout_days,n_features,n_train,n_val,n_test,train_users,val_users,...,val_f1,val_roc_auc,val_pr_auc,val_brier,param_n_estimators,param_max_depth,param_min_samples_leaf,param_learning_rate,param_subsample,param_scale_pos_weight
0,all,4,14,7,427,671,184,339,11,3,...,0.000000,0.989071,0.250000,0.061930,200.0,NaN,1.0,NaN,NaN,NaN
1,all,3,14,0,427,710,184,350,11,3,...,0.000000,0.983607,0.250000,0.067149,500.0,10.0,1.0,NaN,NaN,NaN
2,all,2,14,0,427,713,184,348,11,3,...,0.000000,0.978142,0.200000,0.104294,500.0,6.0,1.0,NaN,NaN,NaN
3,all,3,14,7,427,665,184,338,11,3,...,0.000000,0.907104,0.055556,0.092043,200.0,6.0,1.0,NaN,NaN,NaN
4,all,4,14,0,427,712,184,350,11,3,...,0.000000,0.885246,0.045455,0.112804,500.0,6.0,5.0,NaN,NaN,NaN
5,all,4,3,0,427,962,217,410,13,3,...,0.000000,0.893519,0.035714,0.021538,200.0,10.0,1.0,NaN,NaN,NaN
6,all,4,14,0,427,712,184,350,11,3,...,0.043478,0.846995,0.034483,0.163159,200.0,6.0,NaN,0.05,0.7,auto
7,all,3,14,0,427,710,184,350,11,3,...,0.025974,0.819672,0.029412,0.270627,200.0,6.0,NaN,0.05,0.7,auto
8,all,2,14,7,427,664,184,336,11,3,...,0.000000,0.781421,0.024390,0.092709,500.0,10.0,1.0,NaN,NaN,NaN
9,all,3,7,7,427,777,203,372,13,3,...,0.000000,0.777228,0.021277,0.038021,500.0,10.0,1.0,NaN,NaN,NaN


## 4. Phase 5 — Deep-learning HPO + multi-seed

In [8]:
!python -m scripts.v2.tune_dl_v2 \
    --thresholds {THRESHOLDS} \
    --lengths {LENGTHS} \
    --washouts {WASHOUTS} \
    --seed {SEED} \
    --multi-seeds {MULTI_SEEDS} \
    --epochs {DL_EPOCHS} \
    --batch-size {DL_BATCH}

[v2-tune-dl] outputs -> /content/EECE-693-project/outputs/v2/tables
  thresholds=[2, 3, 4]  lengths=[3, 7, 14]  washouts=[0, 7, 14]
  archs=['gru', 'lstm', 'rnn', 'cnn']  seed=42  epochs=30

[1/3] Building labels, samples, daily feature tables...
  split: train=(939, 294, 113, 808, 343, 562, 917, 278, 328, 867, 190, 701, 454) val=(625, 514, 398) test=(764, 473, 702, 447)

[2/3] Running DL HPO sweep...
  T=2 L=3 W=0 arch=gru: 8 trials
  T=2 L=3 W=0 arch=lstm: 8 trials
  T=2 L=3 W=0 arch=rnn: 8 trials
  T=2 L=3 W=0 arch=cnn: 2 trials
    !! trial failed (cnn {'dropout': 0.2}): RuntimeError('max_pool1d() Invalid computed output size: 0')
    !! trial failed (cnn {'dropout': 0.4}): RuntimeError('max_pool1d() Invalid computed output size: 0')
  T=2 L=3 W=7 arch=gru: 8 trials
  T=2 L=3 W=7 arch=lstm: 8 trials
  T=2 L=3 W=7 arch=rnn: 8 trials
  T=2 L=3 W=7 arch=cnn: 2 trials
    !! trial failed (cnn {'dropout': 0.2}): RuntimeError('max_pool1d() Invalid computed output size: 0')
    !! trial f

In [9]:
ms = pd.read_csv("outputs/v2/tables/tune_dl_v2_multiseed.csv")
ms

,threshold,input_length_days,washout_days,arch,hp,seed,best_epoch,best_val_loss,val_accuracy,val_precision,...,val_roc_auc,val_pr_auc,val_brier,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_pr_auc,test_brier
0,3,14,14,rnn,dropout=0.2;hidden_dim=64;num_layers=2,42,9,0.604425,0.755435,0.021739,...,1.000000,1.000000,0.183566,0.987654,0.0,0.0,0.0,0.900000,0.161219,0.012185
1,3,14,14,rnn,dropout=0.2;hidden_dim=64;num_layers=2,43,13,0.487431,0.836957,0.000000,...,0.781421,0.024390,0.119946,0.987654,0.0,0.0,0.0,0.920312,0.146437,0.012230
2,3,14,14,rnn,dropout=0.2;hidden_dim=64;num_layers=2,44,8,0.364168,0.885870,0.045455,...,0.978142,0.200000,0.106194,0.987654,0.0,0.0,0.0,0.386719,0.012018,0.012246
3,3,14,14,rnn,dropout=0.2;hidden_dim=64;num_layers=2,45,10,0.560335,0.782609,0.024390,...,1.000000,1.000000,0.159523,0.987654,0.0,0.0,0.0,0.653906,0.021131,0.012190
4,3,14,14,rnn,dropout=0.2;hidden_dim=64;num_layers=2,46,10,0.276400,0.934783,0.076923,...,0.972678,0.166667,0.062575,0.987654,0.0,0.0,0.0,0.381250,0.011843,0.012250


## 5. Phase 6 — Sensor ablation

In [13]:
!python -m scripts.v2.sensor_ablation_v2

  reusing split from /content/EECE-693-project/outputs/v2/tables/tune_tabular_v2_split.json
  ablating 81 winners over 9 subsets
  (T, L, W) keys: [(2, 3, 0), (2, 3, 7), (2, 3, 14), (2, 7, 0), (2, 7, 7), (2, 7, 14), (2, 14, 0), (2, 14, 7), (2, 14, 14), (3, 3, 0), (3, 3, 7), (3, 3, 14), (3, 7, 0), (3, 7, 7), (3, 7, 14), (3, 14, 0), (3, 14, 7), (3, 14, 14), (4, 3, 0), (4, 3, 7), (4, 3, 14), (4, 7, 0), (4, 7, 7), (4, 7, 14), (4, 14, 0), (4, 14, 7), (4, 14, 14)]

[1/3] Building labels + samples + daily tables...

[2/3] Pre-building feature tables per (T, L, W, subset)...

[3/3] Refitting winners on each subset...
Wrote 729 ablation rows -> /content/EECE-693-project/outputs/v2/tables/sensor_ablation_v2.csv


In [14]:
abl = pd.read_csv("outputs/v2/tables/sensor_ablation_v2.csv")
abl.sort_values("val_pr_auc", ascending=False).head(30)

,threshold,input_length_days,washout_days,algo,subset,n_features,n_train,n_val,n_test,train_positive,...,val_roc_auc,val_pr_auc,val_brier,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_pr_auc,test_brier
33,3,14,7,rf,no_smartinhaler,400,665,184,338,48,...,1.000000,1.000000,0.100724,0.988166,0.000000,0.000000,0.000000,0.395958,0.011879,0.074159
9,3,14,0,rf,all,427,710,184,350,48,...,0.983607,0.250000,0.067149,0.988571,0.000000,0.000000,0.000000,0.374277,0.010852,0.056550
0,4,14,7,rf,all,427,671,184,339,50,...,0.989071,0.250000,0.061930,0.988201,0.000000,0.000000,0.000000,0.663806,0.031091,0.053512
18,2,14,0,rf,all,427,713,184,348,51,...,0.978142,0.200000,0.104294,0.994253,0.000000,0.000000,0.000000,0.544798,0.009165,0.073205
15,3,14,0,rf,no_smartinhaler,400,710,184,350,48,...,0.967213,0.142857,0.065242,0.988571,0.000000,0.000000,0.000000,0.397399,0.011345,0.051716
16,3,14,0,rf,no_peakflow,355,710,184,350,48,...,0.961749,0.125000,0.061882,0.988571,0.000000,0.000000,0.000000,0.647760,0.021043,0.035915
6,4,14,7,rf,no_smartinhaler,400,671,184,339,50,...,0.926230,0.066667,0.080853,0.988201,0.000000,0.000000,0.000000,0.653731,0.030132,0.059587
200,4,14,14,xgb,smartinhaler_only,86,622,184,325,50,...,0.961749,0.066667,0.033449,0.843077,0.072727,1.000000,0.135593,0.963785,0.163826,0.167793
27,3,14,7,rf,all,427,665,184,338,48,...,0.907104,0.055556,0.092043,0.988166,0.000000,0.000000,0.000000,0.497754,0.013935,0.086173
380,2,14,7,xgb,smartinhaler_only,86,664,184,336,51,...,0.945355,0.055556,0.013813,0.824405,0.000000,0.000000,0.000000,0.696108,0.014333,0.136596


## 6. Phase 7 — Leakage probe (gate before Phase 8)

If any winner's shuffled-label ROC-AUC falls outside `[0.45, 0.55]`,
this prints a WARNING and you should investigate before trusting the
Phase-8 numbers.

In [15]:
!python -m scripts.v2.leakage_probe_v2 --n-shuffles {N_SHUFFLES}

[leakage-probe] 81 winners × 5 shuffles

Building labels + samples + features (all-sensors)...
Wrote 405 probe rows -> /content/EECE-693-project/outputs/v2/tables/leakage_probe_v2.csv
Wrote summary -> /content/EECE-693-project/outputs/v2/tables/leakage_probe_v2_summary.csv
 threshold  input_length_days  washout_days algo  val_roc_auc_mean  val_roc_auc_std  val_pr_auc_mean  val_pr_auc_std  n_shuffles
         2                  3             0   lr          0.448148         0.256796         0.010304        0.005630           5
         2                  3             0   rf          0.551852         0.258253         0.014036        0.009705           5
         2                  3             0  xgb          0.447222         0.284950         0.011663        0.008995           5
         2                  3             7   lr          0.393519         0.299909         0.010165        0.007286           5
         2                  3             7   rf          0.494444         0.3682

## 7. Phase 8 — Final test-set evaluation

In [16]:
!python -m scripts.v2.final_test_eval_v2 \
    --dl-epochs {DL_EPOCHS} \
    --dl-batch-size {DL_BATCH}


 threshold  input_length_days  washout_days algo  val_roc_auc_mean  val_roc_auc_std  val_pr_auc_mean  val_pr_auc_std  n_shuffles
         2                  3             0   lr          0.448148         0.256796         0.010304        0.005630           5
         2                  3             0   rf          0.551852         0.258253         0.014036        0.009705           5
         2                  3             0  xgb          0.447222         0.284950         0.011663        0.008995           5
         2                  3             7   lr          0.393519         0.299909         0.010165        0.007286           5
         2                  3             7  xgb          0.433333         0.438235         0.037584        0.059775           5
         2                  3            14   lr          0.325000         0.138199         0.006997        0.001212           5
         2                  3            14   rf          0.561111         0.382122         0.02

In [17]:
summary = pd.read_csv("outputs/v2/tables/final_test_v2_summary.csv")
summary

,family,name,threshold,input_length_days,washout_days,test_pr_auc,test_roc_auc,test_f1,test_f1_tuned,tuned_threshold,test_precision,test_recall,test_brier,n_test,test_pr_auc_std,n_seeds
0,tabular,rf,4,14,14,0.555180,0.933801,0.000000,0.666667,0.354404,0.000000,0.00,0.047699,325.0,NaN,NaN
1,tabular,xgb,3,14,14,0.526229,0.875000,0.307692,0.066667,0.119929,0.222222,0.50,0.030699,324.0,NaN,NaN
2,tabular,lr,4,14,0,0.022274,0.614884,0.040816,0.017391,0.000645,0.022222,0.25,0.108390,350.0,NaN,NaN
3,deep_learning,rnn,3,14,14,0.261482,0.902188,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,0.148413,5.0


## 8. Phase 9 — Offline analysis (bootstrap CIs, v1-vs-v2, leakage, ablation)

Reads the predictions and summary CSVs from Phase 8 and computes:
- Bootstrap 95% CIs for test PR-AUC / ROC-AUC.
- v1-vs-v2 comparison on matching (T, L, W).
- Leakage-probe overall mean check (should be near 0.5).
- Sensor-ablation top-K per algo.

No training; runs in seconds.

In [18]:
!python -m scripts.v2.analyze_v2 --n-boot 2000


=== v2 offline analysis (/content/EECE-693-project/outputs/v2/tables) ===

[1/4] Bootstrap CIs (tabular winners)...
algo  threshold  input_length_days  washout_days  n_test  n_test_positive   pr_auc  pr_auc_lo95  pr_auc_hi95  roc_auc  roc_auc_lo95  roc_auc_hi95
  rf          4                 14            14     325                4 0.555180     0.033177     1.000000 0.933801      0.801518      1.000000
  rf          3                 14            14     324                4 0.554427     0.024715     1.000000 0.916406      0.726772      1.000000
 xgb          3                 14            14     324                4 0.526229     0.017699     1.000000 0.875000      0.632547      1.000000
  rf          2                 14            14     322                2 0.509615     0.008850     1.000000 0.840625      0.646875      1.000000
 xgb          2                 14            14     322                2 0.504545     0.004386     1.000000 0.659375      0.286604      1.000000
 xgb   

In [19]:
from pathlib import Path
tbl = Path("outputs/v2/tables")
for name in ["analysis_v2_tabular_ci.csv", "analysis_v2_dl_ci.csv", "analysis_v2_v1_vs_v2.csv", "analysis_v2_sensor_ablation_topk.csv"]:
    pth = tbl / name
    if pth.exists():
        print(f"\n--- {name} ---")
        display(pd.read_csv(pth))


--- analysis_v2_tabular_ci.csv ---


,algo,threshold,input_length_days,washout_days,n_test,n_test_positive,pr_auc,pr_auc_lo95,pr_auc_hi95,roc_auc,roc_auc_lo95,roc_auc_hi95
0,rf,4,14,14,325,4,0.555180,0.033177,1.000000,0.933801,0.801518,1.000000
1,rf,3,14,14,324,4,0.554427,0.024715,1.000000,0.916406,0.726772,1.000000
2,xgb,3,14,14,324,4,0.526229,0.017699,1.000000,0.875000,0.632547,1.000000
3,rf,2,14,14,322,2,0.509615,0.008850,1.000000,0.840625,0.646875,1.000000
4,xgb,2,14,14,322,2,0.504545,0.004386,1.000000,0.659375,0.286604,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
76,lr,2,7,7,370,3,0.007999,0.003817,0.020241,0.337875,0.276423,0.411924
77,lr,2,3,7,394,3,0.007439,0.003226,0.018614,0.346974,0.203562,0.548469
78,lr,2,14,14,322,2,0.007051,0.004255,0.020515,0.354687,0.261682,0.448598
79,lr,2,14,7,336,2,0.006366,0.003356,0.018519,0.312874,0.107463,0.533840



--- analysis_v2_dl_ci.csv ---


,arch,threshold,input_length_days,washout_days,n_seeds,pr_auc_mean,pr_auc_std,roc_auc_mean,roc_auc_std
0,rnn,3,14,14,5,0.261482,0.148413,0.902188,0.057088



--- analysis_v2_v1_vs_v2.csv ---


,algo,threshold,input_length_days,washout_days,v1_best_pr_auc,v1_best_roc_auc,v2_pr_auc,v2_pr_auc_lo95,v2_pr_auc_hi95,v2_roc_auc,v2_roc_auc_lo95,v2_roc_auc_hi95,pr_auc_delta,roc_auc_delta
0,rf,4,14,14,NaN,NaN,0.555180,0.033177,1.000000,0.933801,0.801518,1.000000,NaN,NaN
1,rf,3,14,14,NaN,NaN,0.554427,0.024715,1.000000,0.916406,0.726772,1.000000,NaN,NaN
2,xgb,3,14,14,NaN,NaN,0.526229,0.017699,1.000000,0.875000,0.632547,1.000000,NaN,NaN
3,rf,2,14,14,NaN,NaN,0.509615,0.008850,1.000000,0.840625,0.646875,1.000000,NaN,NaN
4,xgb,2,14,14,NaN,NaN,0.504545,0.004386,1.000000,0.659375,0.286604,1.000000,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76,lr,2,7,7,NaN,NaN,0.007999,0.003817,0.020241,0.337875,0.276423,0.411924,NaN,NaN
77,lr,2,3,7,NaN,NaN,0.007439,0.003226,0.018614,0.346974,0.203562,0.548469,NaN,NaN
78,lr,2,14,14,NaN,NaN,0.007051,0.004255,0.020515,0.354687,0.261682,0.448598,NaN,NaN
79,lr,2,14,7,NaN,NaN,0.006366,0.003356,0.018519,0.312874,0.107463,0.533840,NaN,NaN



--- analysis_v2_sensor_ablation_topk.csv ---


,algo,subset,threshold,input_length_days,washout_days,val_pr_auc,val_roc_auc,val_f1,test_pr_auc,test_roc_auc,test_f1
0,lr,smartinhaler_only,3,14,0,0.035714,0.890710,0.068966,0.018405,0.596098,0.040000
1,lr,smartinhaler_only,4,14,0,0.035714,0.890710,0.068966,0.024151,0.670520,0.033333
2,lr,smartinhaler_only,2,14,0,0.035714,0.890710,0.068966,0.007887,0.465318,0.000000
3,rf,no_smartinhaler,3,14,7,1.000000,1.000000,0.400000,0.011879,0.395958,0.000000
4,rf,all,4,14,7,0.250000,0.989071,0.000000,0.031091,0.663806,0.000000
5,rf,all,3,14,0,0.250000,0.983607,0.000000,0.010852,0.374277,0.000000
6,xgb,smartinhaler_only,4,14,14,0.066667,0.961749,0.111111,0.163826,0.963785,0.135593
7,xgb,smartinhaler_only,2,14,7,0.055556,0.945355,0.000000,0.014333,0.696108,0.000000
8,xgb,smartinhaler_only,4,14,7,0.052632,0.939891,0.000000,0.058660,0.746269,0.060606


## 9. (Optional) Copy v2 outputs back to Drive

Useful when Colab's local FS will be wiped at session end.

In [20]:
import shutil
drive_target = Path("/content/drive/MyDrive/AAMOS/outputs/v2")
if drive_target.parent.exists():
    drive_target.mkdir(parents=True, exist_ok=True)
    shutil.copytree("outputs/v2", drive_target, dirs_exist_ok=True)
    print("copied -> ", drive_target)
else:
    print("Drive output folder not present; skipping copy.")

copied ->  /content/drive/MyDrive/AAMOS/outputs/v2
